# Phase 7 — CausalMask Metrics on Frozen Baseline Models

**Objective:** Measure lesion necessity, lesion sufficiency, and background
invariance using the frozen baseline models from Phase 5. Do not retrain.

**Rules:**
1. Use the original predicted class as the primary decision-faithfulness target.
2. Also compute true-class versions only for correctly classified samples.
3. Keep predicted-class and true-class metrics in clearly separate columns.
4. Implement raw + normalized necessity, sufficiency, invariance, flip rate,
   donor-stratified invariance, lesion-vs-sham difference.
5. Use multiple deterministic donors per sample.
6. Evaluate margins 0%, 10%, 20%; Telea and NS removal; lesion + sham.
7. Save one row per run, fold, sample, target, margin, operator, donor.
8. Report denominators and failure counts.
9. Group-aware bootstrap confidence intervals.
10. Validate components before composite; then harmonic/arithmetic/geometric.
11. Do not select aggregation weights using observed performance.
12. No model retrained; no external data used.

**Status labels:** `planned` | `implemented` | `runnable` | `executed` |
`validated` | `failed` | `blocked`

**Phase 7 gate:**
- Component metrics pass synthetic tests.
- Sham controls are included.
- Intervention-operator sensitivity is reported.
- The composite score remains secondary.
- No model was retrained.
- No external data were used.

## 7.0 — Colab bootstrap

In [ ]:
import os
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/Sayem7456/CausalMask-XAI.git"
COLAB_TARGET = Path("/content/CausalMask-XAI")

if IN_COLAB:
    print("Detected Google Colab environment.")
    if COLAB_TARGET.exists() and (COLAB_TARGET / "CausalMask-XAI.md").exists():
        print(f"Repository present at {COLAB_TARGET}. Pulling latest...")
        !cd {COLAB_TARGET} && git pull --ff-only
        print("Repository updated to latest commit.")
    else:
        if COLAB_TARGET.exists():
            import shutil
            shutil.rmtree(COLAB_TARGET)
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {COLAB_TARGET}
        assert (COLAB_TARGET / "CausalMask-XAI.md").exists(), "Clone failed"
    os.environ["CAUSALMASK_PROJECT_ROOT"] = str(COLAB_TARGET)
    !cd {COLAB_TARGET} && pip install -e .[dev] --quiet 2>&1 | tail -3
    print("Package installed in editable mode.")
else:
    print("Not in Colab — skipping bootstrap.")

## 7.1 — Project root, deterministic seeds, environment

In [ ]:
import sys
from pathlib import Path


def _resolve_project_root() -> Path:
    env_root = os.environ.get("CAUSALMASK_PROJECT_ROOT")
    if env_root:
        p = Path(env_root)
        if (p / "CausalMask-XAI.md").exists():
            return p.resolve()
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "CausalMask-XAI.md").exists():
            return candidate.resolve()
    fallback = Path("/content/CausalMask-XAI")
    if fallback.exists() and (fallback / "CausalMask-XAI.md").exists():
        return fallback.resolve()
    raise RuntimeError(
        "Cannot resolve project root. "
        "Set CAUSALMASK_PROJECT_ROOT or run from within the repo."
    )


PROJECT_ROOT = _resolve_project_root()
print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "CausalMask-XAI.md").exists(), "Marker file missing"

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from datetime import datetime, timezone
import json
import torch
import numpy as np

from causalmask.reproducibility import configure_reproducibility, capture_environment

SEED = 42
rep_info = configure_reproducibility(SEED)
print(f"Reproducibility configured: seed={SEED}")
for k, v in rep_info.items():
    print(f"  {k}: {v}")

env_info = capture_environment(PROJECT_ROOT)
print(f"\nEnvironment:")
print(f"  Python: {env_info['python']}")
print(f"  Torch: {env_info['torch']}")
print(f"  CUDA available: {env_info['cuda_available']}")
DEVICE = "cuda" if env_info["cuda_available"] else "cpu"
print(f"  Device: {DEVICE}")

## 7.2 — Display active configuration

In [ ]:
PHASE_CONFIG = {
    "phase": "07",
    "phase_name": "CausalMask Metrics on Frozen Baseline Models",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "backbone": "efficientnet_b0",
    "pretrained_weight_id": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "num_classes": 2,
    "input_size": [224, 224],
    "margins": [0.0, 0.1, 0.2],
    "removal_operators": ["telea", "navier_stokes"],
    "donor_classes": ["same", "opposite"],
    "n_donors_per_sample": 3,
    "blur_sigma": 20.0,
    "target_definitions": ["predicted", "true"],
    "bootstrap_n": 2_000,
    "manifest_version": "v1",
    "split_name": "busi_binary_grouped_5fold_v1",
    "external_datasets": ["bus_uclm"],
    "bus_uclm_frozen": True,
    "n_folds": 5,
    "fold_run_ids": {
        "fold_0": "baseline_ce_effb0_fold0_seed42",
        "fold_1": "baseline_ce_effb0_fold1_seed42",
        "fold_2": "baseline_ce_effb0_fold2_seed42",
        "fold_3": "baseline_ce_effb0_fold3_seed42",
        "fold_4": "baseline_ce_effb0_fold4_seed42",
    },
    "datasets": {
        "busi": {
            "archive_rel": "data/raw/archives/breast-ultrasound-images-dataset.zip",
            "extract_rel": "data/raw/extracted/busi",
        },
        "bus_uclm": {
            "archive_rel": "data/raw/archives/bus-uclm-breast-ultrasound-dataset.zip",
            "extract_rel": "data/raw/extracted/bus_uclm",
        },
    },
    "experiment_note": (
        "CausalMask component metrics on frozen baseline models. "
        "No retraining. No external data. Composite score secondary."
    ),
}

print(json.dumps(PHASE_CONFIG, indent=2, default=str))

## 7.3 — Mount Drive & restore Phase 2/3/5/6 artifacts

In [ ]:
import shutil
import zipfile

MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
REPORTS_DIR = PROJECT_ROOT / "reports"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"
PHASES_DIR = PROJECT_ROOT / "artifacts" / "phases"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"
ARCHIVES_DIR = PROJECT_ROOT / "data" / "raw" / "archives"
EXTRACT_DIR = PROJECT_ROOT / "data" / "raw" / "extracted"
PLOTS_DIR = RESULTS_DIR / "baseline_causalmask_plots"

for d in [MANIFESTS_DIR, SPLITS_DIR, RESULTS_DIR, PHASES_DIR,
          RUNS_DIR, ARCHIVES_DIR, EXTRACT_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Mount Google Drive
DRIVE_BASE = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/CausalMask-XAI")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will sync to {DRIVE_BASE}")
    for subdir in ["manifests", "splits", "reports", "artifacts", "runs"]:
        (DRIVE_BASE / subdir).mkdir(parents=True, exist_ok=True)
else:
    print("Not in Colab — Drive not mounted. Artifacts saved locally only.")


def restore_from_drive(subdir: str, filename: str, local_dir: Path) -> bool:
    if DRIVE_BASE is None:
        return False
    src = DRIVE_BASE / subdir / filename
    dst = local_dir / filename
    if dst.exists():
        return False
    if not src.exists():
        print(f"  [WARN] Not found on Drive: {src}")
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Restored from Drive: {dst}")
    return True


def save_to_drive(src: Path, subdir: str) -> bool:
    if DRIVE_BASE is None:
        return False
    dst = DRIVE_BASE / subdir / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Synced to Drive: {dst}")
    return True


def save_dir_to_drive(src_dir: Path, subdir: str) -> int:
    """Recursively copy a directory tree to Drive. Returns file count."""
    if DRIVE_BASE is None:
        return 0
    dst_base = DRIVE_BASE / subdir / src_dir.name
    count = 0
    for f in src_dir.rglob("*"):
        if f.is_file():
            rel = f.relative_to(src_dir)
            dst = dst_base / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            count += 1
    if count > 0:
        print(f"  Synced {count} files to Drive: {dst_base}")
    return count


print("\n--- Restoring Phase 2/3 artifacts from Drive ---")
for fname in [
    f"busi_manifest_{PHASE_CONFIG['manifest_version']}.parquet",
    f"busi_manifest_summary_{PHASE_CONFIG['manifest_version']}.json",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

for fname in [
    "busi_manifest_v2_grouped.parquet",
    "duplicate_clusters_v1.parquet",
    "duplicate_candidates_v1.parquet",
]:
    restore_from_drive("manifests", fname, MANIFESTS_DIR)

restore_from_drive("splits",
                   f"{PHASE_CONFIG['split_name']}.json", SPLITS_DIR)

for ds_name, cfg in PHASE_CONFIG.get("datasets", {}).items():
    archive_path = ARCHIVES_DIR / Path(cfg["archive_rel"]).name
    restore_from_drive("archives", archive_path.name, ARCHIVES_DIR)

print("
--- Restoring Phase 5 baseline checkpoints from Drive ---")

# Phase 5 uses save_dir_to_drive which preserves run_id subdirectory
#    runs/baseline_ce_effb0_fold0_seed42/checkpoints/best.pt
for fold_name, run_id in PHASE_CONFIG["fold_run_ids"].items():
    ckpt_filename = "best.pt"
    local_dst = RUNS_DIR / run_id / "checkpoints" / ckpt_filename
    if DRIVE_BASE is not None:
        src = DRIVE_BASE / "runs" / run_id / "checkpoints" / ckpt_filename
        if src.exists():
            local_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local_dst)
            print(f"  Restored {run_id}/{ckpt_filename} from Drive")
        else:
            print(f"  [WARN] Checkpoint not found: {src}")
    else:
        print(f"  [SKIP] {run_id} — Drive not mounted")

import hashlib
# Verify checkpoints are unique per fold
ckpt_hashes_local = {}
for fold_name, run_id in PHASE_CONFIG["fold_run_ids"].items():
    ckpt = RUNS_DIR / run_id / "checkpoints" / "best.pt"
    if ckpt.exists():
        sha = hashlib.sha256(ckpt.read_bytes()).hexdigest()
        ckpt_hashes_local[run_id] = sha
unique_ckpts = len(set(ckpt_hashes_local.values()))
print(f"  Restored {len(ckpt_hashes_local)} of 5 checkpoints ({unique_ckpts} unique)")
if unique_ckpts < len(ckpt_hashes_local) and len(ckpt_hashes_local) > 1:
    print(f"  [WARN] Duplicate checkpoints detected across folds.")
print("\n--- Extracting archives if needed ---")
for ds_name, cfg in PHASE_CONFIG.get("datasets", {}).items():
    extract_path = PROJECT_ROOT / cfg["extract_rel"]
    archive = ARCHIVES_DIR / Path(cfg["archive_rel"]).name
    if extract_path.exists() and any(extract_path.iterdir()):
        print(f"  {ds_name}: extracted data present.")
    elif archive.exists():
        print(f"  {ds_name}: extracting from {archive}...")
        extract_path.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(extract_path)
        print(f"  {ds_name}: extracted to {extract_path}")
    else:
        print(f"  {ds_name}: no archive, no extracted data — blocked.")

print("--- Restore complete ---\n")

## 7.4 — Load split, manifest, and build test-sample index

In [ ]:
import pandas as pd

manifest_version = PHASE_CONFIG["manifest_version"]
manifest_name = f"busi_manifest_{manifest_version}"
grouped_manifest_path = MANIFESTS_DIR / "busi_manifest_v2_grouped.parquet"

if grouped_manifest_path.exists():
    manifest_df = pd.read_parquet(grouped_manifest_path)
    print(f"Loaded grouped manifest: {len(manifest_df)} samples")
else:
    manifest_path = MANIFESTS_DIR / f"{manifest_name}.parquet"
    manifest_df = pd.read_parquet(manifest_path)
    print(f"Loaded manifest: {len(manifest_df)} samples")

split_path = SPLITS_DIR / f"{PHASE_CONFIG['split_name']}.json"
with open(split_path) as f:
    split_data = json.load(f)

from causalmask.data.splits import compute_manifest_digest
manifest_digest = compute_manifest_digest(manifest_df)
split_digest = split_data["metadata"]["split_digest"]
print(f"Split digest: {split_digest}")
print(f"Manifest digest: {manifest_digest}")

# Verify digests match Phase 5 registered values
EXPECTED_SPLIT_DIGEST = "2a88e7ada1aff73e245d6d8b48693aaebb45ce5ad7568f6753f25cce4935f151"
EXPECTED_MANIFEST_DIGEST = "6462d283b3fcfe6657ece48daa8b7a0b09dc786bbfb34ed990c7d4d904f84304"
assert split_digest == EXPECTED_SPLIT_DIGEST, (
    f"Split digest mismatch! Expected {EXPECTED_SPLIT_DIGEST[:12]}..., "
    f"got {split_digest[:12]}..."
)
assert manifest_digest == EXPECTED_MANIFEST_DIGEST, (
    f"Manifest digest mismatch! Expected {EXPECTED_MANIFEST_DIGEST[:12]}..., "
    f"got {manifest_digest[:12]}..."
)
print("Digest verification passed — split and manifest match Phase 5.")

# Build per-fold test-sample index
fold_test_samples = {}
all_test_samples = set()
for fold_key in sorted(split_data["folds"].keys()):
    test_ids = split_data["folds"][fold_key]["test"]
    fold_test_samples[fold_key] = test_ids
    all_test_samples.update(test_ids)
    print(f"  {fold_key}: {len(test_ids)} test samples")
print(f"Total unique test samples: {len(all_test_samples)}")

# Filter manifest to test samples only
test_manifest = manifest_df[
    manifest_df["sample_id"].isin(all_test_samples)
].copy()
print(f"Test manifest: {len(test_manifest)} samples")
print(f"  benign: {(test_manifest['normalized_label'] == 'benign').sum()}")
print(f"  malignant: {(test_manifest['normalized_label'] == 'malignant').sum()}")

# Check if BUSI extracted data exists
busi_extract = EXTRACT_DIR / "busi"
USE_REAL_DATA = busi_extract.exists() and any(busi_extract.iterdir())
print(f"\nReal BUSI data available: {USE_REAL_DATA}")
if USE_REAL_DATA:
    print(f"  BUSI extract: {busi_extract}")

## 7.5 — Load frozen baseline models and preprocessing

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms as T
from causalmask.models.factory import create_model

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

fold_models = {}
for fold_name, run_id in PHASE_CONFIG["fold_run_ids"].items():
    ckpt_path = RUNS_DIR / run_id / "checkpoints" / "best.pt"
    if not ckpt_path.exists():
        print(f"  [SKIP] {fold_name}: checkpoint not found at {ckpt_path}")
        continue
    model = create_model("efficientnet_b0", num_classes=2, pretrained=False)
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state.get("model_state_dict", state))
    model.to(DEVICE)
    model.eval()
    fold_models[fold_name] = model
    print(f"  Loaded {fold_name}: {run_id}/best.pt")

models_loaded = list(fold_models.keys())
print(f"\nLoaded {len(models_loaded)} of {len(PHASE_CONFIG['fold_run_ids'])} fold models")

## 7.6 — Counterfactual generator and inference helpers

In [ ]:
from PIL import Image
import cv2

from causalmask.counterfactuals.masks import MarginConfig, lesion_plus_margin
from causalmask.counterfactuals.sufficient import (
    SufficientConfig, generate_lesion_sufficient,
)
from causalmask.counterfactuals.removal import (
    RemovalConfig, RemovalOperator, generate_lesion_removed,
)
from causalmask.counterfactuals.background_swap import (
    SwapConfig, generate_background_swap, _select_donor,
)
from causalmask.counterfactuals.controls import (
    ControlsConfig, generate_random_region_removal,
)


def load_image_uint8(image_path: str) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    return np.array(img, dtype=np.uint8)


def load_mask_uint8(mask_path) -> np.ndarray:
    if mask_path is None or not Path(mask_path).exists():
        return None
    mask = Image.open(mask_path).convert("L")
    return (np.array(mask) > 127).astype(np.uint8)


def infer_model(model, image_uint8: np.ndarray) -> np.ndarray:
    """Return softmax probabilities [n_classes] as float32 numpy."""
    pil_img = Image.fromarray(image_uint8)
    tensor = eval_transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    return probs.astype(np.float64)


def build_donor_pool(manifest_df, partition_ids, rng_seed=42):
    """Build a donor pool dict keyed by sample_id."""
    partition = manifest_df[manifest_df["sample_id"].isin(partition_ids)]
    candidates = []
    for _, row in partition.iterrows():
        candidates.append({
            "sample_id": row["sample_id"],
            "normalized_label": row["normalized_label"],
            "image_path": row["image_path"],
        })
    return candidates

print("Counterfactual helpers defined.")

## 7.7 — Main evaluation loop: counterfactuals + inference + metrics

In [ ]:
from causalmask.evaluation.faithfulness import compute_per_sample_causal_metrics

MARGINS = PHASE_CONFIG["margins"]
N_DONORS = PHASE_CONFIG["n_donors_per_sample"]
all_records = []
failed_samples = []
n_processed = 0
n_failed = 0

for fold_name, model in fold_models.items():
    fold_idx = fold_name.split("_")[1]
    test_ids = set(fold_test_samples.get(fold_name, []))
    train_ids = set(split_data["folds"][fold_name].get("train", []))
    run_id = PHASE_CONFIG["fold_run_ids"].get(fold_name, f"unknown_{fold_name}")

    if not test_ids:
        print(f"  [{fold_name}] No test samples — skipping")
        continue

    donor_pool = build_donor_pool(manifest_df, list(train_ids), rng_seed=SEED)

    fold_samples = test_manifest[test_manifest["sample_id"].isin(test_ids)]
    print(f"\n[{fold_name}] {len(fold_samples)} test samples")

    for _, row in fold_samples.iterrows():
        sample_id = row["sample_id"]
        image_path = row.get("image_path")
        mask_path = row.get("mask_path")
        true_label_str = row["normalized_label"]
        true_class_idx = 1 if true_label_str == "malignant" else 0
        group_id = row.get("group_id", sample_id)

        if not USE_REAL_DATA or not Path(image_path).exists():
            n_failed += 1
            continue

        try:
            image_uint8 = load_image_uint8(image_path)
            mask_uint8 = load_mask_uint8(mask_path)
            if mask_uint8 is None:
                n_failed += 1
                failed_samples.append({
                    "sample_id": sample_id, "fold": fold_name,
                    "reason": "no_mask",
                })
                continue

            p_original = infer_model(model, image_uint8)
            predicted_class = int(np.argmax(p_original))
            p_orig_conf = float(p_original[predicted_class])

            # Generate counterfactuals for each margin + operator
            for margin_ratio in MARGINS:
                margin_cfg = MarginConfig(margin_ratio=margin_ratio)

                # --- Lesion sufficient ---
                suff_img, mplus = generate_lesion_sufficient(
                    image_uint8, mask_uint8,
                    SufficientConfig(margin_config=margin_cfg, blur_sigma=20.0),
                )
                p_sufficient = infer_model(model, suff_img)

                # --- Lesion removed (Telea + Navier-Stokes) ---
                p_removed_t = None
                p_removed_ns = None
                for op_name in PHASE_CONFIG["removal_operators"]:
                    op = RemovalOperator.TELEA if op_name == "telea" else RemovalOperator.NAVIER_STOKES
                    removed_img, _ = generate_lesion_removed(
                        image_uint8, mask_uint8,
                        RemovalConfig(margin_config=margin_cfg, operator=op),
                    )
                    p_removed = infer_model(model, removed_img)
                    if op_name == "telea":
                        p_removed_t = p_removed
                    else:
                        p_removed_ns = p_removed

                # --- Background swaps (same + opposite donors) ---
                p_swaps_same = []
                p_swaps_opposite = []
                for donor_class in PHASE_CONFIG["donor_classes"]:
                    swap_cfg = SwapConfig(
                        margin_config=margin_cfg,
                        donor_class=donor_class,
                        seed=SEED + int(fold_idx),
                    )
                    for donor_i in range(N_DONORS):
                        donor_seed = SEED * 1000 + int(fold_idx) * 100 + donor_i
                        donor_rng = np.random.default_rng(donor_seed)
                        donor = _select_donor(
                            sample_id, true_label_str, donor_pool,
                            SwapConfig(donor_class=donor_class, seed=donor_seed),
                            donor_rng,
                        )
                        if donor is None:
                            continue
                        donor_img = load_image_uint8(donor["image_path"])
                        swap_img, _ = generate_background_swap(
                            image_uint8, mask_uint8, donor_img,
                            swap_cfg,
                        )
                        p_swap = infer_model(model, swap_img)
                        if donor_class == "same":
                            p_swaps_same.append(p_swap)
                        else:
                            p_swaps_opposite.append(p_swap)

                # --- Sham control (same-area random region removal) ---
                sham_img, sham_mask, _ = generate_random_region_removal(
                    image_uint8, mask_uint8,
                    ControlsConfig(seed=SEED * 100 + int(fold_idx)),
                )
                p_sham = infer_model(model, sham_img)

                # --- Compute metrics ---
                metrics = compute_per_sample_causal_metrics(
                    p_original=p_original,
                    p_sufficient=p_sufficient,
                    p_removed_telea=p_removed_t,
                    p_removed_navier=p_removed_ns,
                    p_swaps_same=p_swaps_same,
                    p_swaps_opposite=p_swaps_opposite,
                    p_sham_removed=p_sham,
                    true_class=true_class_idx,
                )

                record = {
                    "run_id": run_id,
                    "fold": int(fold_idx),
                    "sample_id": sample_id,
                    "group_id": group_id,
                    "true_label": true_label_str,
                    "true_class": true_class_idx,
                    "original_confidence": p_orig_conf,
                    "margin": margin_ratio,
                    "n_same_donors": len(p_swaps_same),
                    "n_opposite_donors": len(p_swaps_opposite),
                }
                record.update(metrics)
                all_records.append(record)

            n_processed += 1

        except Exception as e:
            n_failed += 1
            failed_samples.append({
                "sample_id": sample_id, "fold": fold_name,
                "reason": str(e)[:200],
            })

        if n_processed % 50 == 0:
            print(f"    Processed {n_processed} samples...")

print(f"\n=== Main loop complete ===")
print(f"  Processed: {n_processed}")
print(f"  Failed: {n_failed}")
print(f"  Records: {len(all_records)}")

## 7.8 — Save detailed results parquet

In [ ]:
if all_records:
    # Generate versioned run ID to prevent overwriting prior results
    run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_version = f"phase07_{run_ts}_seed{SEED}"
    run_results_dir = RESULTS_DIR / run_version
    run_results_dir.mkdir(parents=True, exist_ok=False)
    
    results_df = pd.DataFrame(all_records)
    results_path = run_results_dir / "baseline_causal_components.parquet"
    results_df.to_parquet(results_path, index=False)
    print(f"Saved {len(results_df)} records to {results_path}")
    print(f"Columns: {list(results_df.columns)}")
    
    # Save failed samples
    if failed_samples:
        failed_df = pd.DataFrame(failed_samples)
        failed_path = run_results_dir / "failed_samples.parquet"
        failed_df.to_parquet(failed_path, index=False)
        print(f"Saved {len(failed_df)} failed-sample records to {failed_path}")
    
    # Also save as latest for convenience (with atomic write)
    latest_path = RESULTS_DIR / "baseline_causal_components.parquet"
    results_df.to_parquet(latest_path, index=False)
    save_to_drive(results_path, "reports/results")
    save_to_drive(latest_path, "reports/results")
else:
    print("No results to save.")
    results_df = pd.DataFrame()
    run_results_dir = None

## 7.9 — Group-aware bootstrap confidence intervals

In [ ]:
from causalmask.statistics.bootstrap import group_aware_bootstrap_ci

bootstrap_results = {}

if results_df is not None and len(results_df) > 0:
    metric_columns = [
        col for col in results_df.columns
        if col.startswith("predicted_") or col.startswith("true_")
    ]
    metric_columns = [
        c for c in metric_columns
        if results_df[c].dtype in ("float64", "float32", "float16", "int64")
        and not c.endswith("_n_donors")
        and results_df[c].notna().any()
    ]

    for col in sorted(metric_columns):
        valid_mask = results_df[col].notna()
        values = results_df.loc[valid_mask, col].values
        groups = results_df.loc[valid_mask, "group_id"].values
        if len(values) < 5:
            continue
        ci = group_aware_bootstrap_ci(
            values, groups,
            n_bootstrap=PHASE_CONFIG["bootstrap_n"],
            seed=SEED,
        )
        bootstrap_results[col] = ci

    print(f"Bootstrap CIs computed for {len(bootstrap_results)} metrics")

    # Print summary table
    print(f"\n{'Metric':<50} {'Mean':>8} {'CI Lower':>8} {'CI Upper':>8} {'N':>6}")
    print("-" * 85)
    for col, ci in sorted(bootstrap_results.items()):
        print(f"{col:<50} {ci['point_estimate']:8.4f} "
              f"{ci['ci_lower']:8.4f} {ci['ci_upper']:8.4f} {ci['n_valid']:6d}")
else:
    print("No results available for bootstrap.")

## 7.10 — CausalMask composite scores and aggregation sensitivity

In [ ]:
from causalmask.evaluation.causalmask_score import (
    compute_causalmask_harmonic,
    compute_causalmask_arithmetic,
    compute_causalmask_geometric,
    compute_aggregation_sensitivity,
)

composite_scores = []
aggregation_sensitivity = {}

if results_df is not None and len(results_df) > 0:
    # Aggregate per-sample across margins and operators (use Telea, margin=10% as primary)
    primary_mask = (
        (results_df["margin"] == 0.1)
        & results_df["predicted_norm_necessity_telea"].notna()
        & results_df["predicted_sufficiency"].notna()
        & results_df["predicted_background_invariance"].notna()
    )
    primary_df = results_df[primary_mask].copy()

    if len(primary_df) > 0:
        n_vals = primary_df["predicted_norm_necessity_telea"].values
        s_vals = primary_df["predicted_sufficiency"].values
        b_vals = primary_df["predicted_background_invariance"].values

        valid = np.isfinite(n_vals) & np.isfinite(s_vals) & np.isfinite(b_vals)
        n_vals = n_vals[valid]
        s_vals = s_vals[valid]
        b_vals = b_vals[valid]

        for i in range(len(n_vals)):
            composite_scores.append({
                "harmonic": compute_causalmask_harmonic(n_vals[i], s_vals[i], b_vals[i]),
                "arithmetic": compute_causalmask_arithmetic(n_vals[i], s_vals[i], b_vals[i]),
                "geometric": compute_causalmask_geometric(n_vals[i], s_vals[i], b_vals[i]),
                "necessity": float(n_vals[i]),
                "sufficiency": float(s_vals[i]),
                "background_invariance": float(b_vals[i]),
            })

        composite_df = pd.DataFrame(composite_scores)
        print(f"Composite scores computed for {len(composite_df)} samples")
        print(f"\nAggregation comparison (mean):")
        for agg in ["harmonic", "arithmetic", "geometric"]:
            vals = composite_df[agg].dropna()
            print(f"  {agg:<16}: {vals.mean():.4f} ± {vals.std():.4f}  (median {vals.median():.4f})")

        # Aggregation sensitivity
        aggregation_sensitivity = compute_aggregation_sensitivity(n_vals, s_vals, b_vals)
        print(f"\nSpearman rank correlations:")
        for key, val in aggregation_sensitivity.get("spearman_correlations", {}).items():
            print(f"  {key}: {val:.4f}")
    else:
        print("No primary-margin samples for composite score.")
else:
    print("No results for composite score.")

## 7.11 — Distribution analysis by subgroup

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

distribution_data = {}

if results_df is not None and len(results_df) > 0:
    margin_10_df = results_df[results_df["margin"] == 0.1]

    def _bootstrap_mean_vals(values, groups, n_bs=200):
        ci = group_aware_bootstrap_ci(values, groups, n_bootstrap=n_bs, seed=SEED)
        return ci["point_estimate"], ci["ci_lower"], ci["ci_upper"]

    # 1. Benign vs Malignant
    for label in ["benign", "malignant"]:
        sub = margin_10_df[margin_10_df["true_label"] == label]
        for col in ["predicted_norm_necessity_telea", "predicted_sufficiency", "predicted_background_invariance"]:
            vals = sub[col].dropna().values
            grps = sub.loc[sub[col].notna(), "group_id"].values
            if len(vals) > 0:
                est, lo, hi = _bootstrap_mean_vals(vals, grps)
                distribution_data[f"{label}_{col}"] = {"mean": est, "ci_lower": lo, "ci_upper": hi, "n": len(vals)}

    # 2. Correct vs Incorrect
    for correct in [True, False]:
        sub = margin_10_df[margin_10_df["is_correct"] == correct]
        label = "correct" if correct else "incorrect"
        for col in ["predicted_norm_necessity_telea", "predicted_sufficiency", "predicted_background_invariance"]:
            vals = sub[col].dropna().values
            grps = sub.loc[sub[col].notna(), "group_id"].values
            if len(vals) > 0:
                est, lo, hi = _bootstrap_mean_vals(vals, grps)
                distribution_data[f"{label}_{col}"] = {"mean": est, "ci_lower": lo, "ci_upper": hi, "n": len(vals)}

    # 3. Margin sensitivity
    for margin in PHASE_CONFIG["margins"]:
        sub = results_df[(results_df["margin"] == margin)
                          & results_df["predicted_norm_necessity_telea"].notna()]
        for col in ["predicted_norm_necessity_telea", "predicted_sufficiency"]:
            vals = sub[col].dropna().values
            grps = sub.loc[sub[col].notna(), "group_id"].values
            if len(vals) > 0:
                est, lo, hi = _bootstrap_mean_vals(vals, grps)
                distribution_data[f"margin{margin}_{col}"] = {"mean": est, "ci_lower": lo, "ci_upper": hi, "n": len(vals)}

    # 4. Lesion vs Sham
    sham_col = "predicted_lesion_vs_sham_diff_telea"
    if sham_col in margin_10_df.columns:
        sub = margin_10_df[margin_10_df[sham_col].notna()]
        vals = sub[sham_col].values
        grps = sub["group_id"].values
        if len(vals) > 0:
            est, lo, hi = _bootstrap_mean_vals(vals, grps)
            distribution_data["lesion_vs_sham_difference"] = {"mean": est, "ci_lower": lo, "ci_upper": hi, "n": len(vals)}

    # Print table
    print(f"{'Subgroup':<45} {'Mean':>8} {'CI Low':>8} {'CI High':>8} {'N':>6}")
    print("-" * 80)
    for key, d in sorted(distribution_data.items()):
        print(f"{key:<45} {d['mean']:8.4f} {d['ci_lower']:8.4f} {d['ci_upper']:8.4f} {d['n']:6d}")

    # Generate margin sensitivity plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    component_cols = [
        ("predicted_norm_necessity_telea", "Lesion Necessity (norm)"),
        ("predicted_sufficiency", "Lesion Sufficiency"),
        ("predicted_background_invariance", "Background Invariance"),
    ]
    for ax, (col, title) in zip(axes, component_cols):
        margins_plot = []
        means = []
        cis = []
        for margin in PHASE_CONFIG["margins"]:
            sub = results_df[(results_df["margin"] == margin)
                              & results_df[col].notna()]
            if len(sub) > 0:
                margins_plot.append(margin)
                est, lo, hi = _bootstrap_mean_vals(
                    sub[col].values, sub["group_id"].values)
                means.append(est)
                cis.append((est - lo, hi - est))
        if means:
            err_lo, err_hi = zip(*cis)
            ax.errorbar(margins_plot, means, yerr=[err_lo, err_hi],
                        fmt="o-", capsize=5)
        ax.set_title(title)
        ax.set_xlabel("Margin ratio")
        ax.set_ylabel("Score")
    plt.tight_layout()
    plots_save = PLOTS_DIR / "margin_sensitivity.png"
    fig.savefig(plots_save, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nMargin sensitivity plot saved: {plots_save}")
    save_to_drive(plots_save, "reports/results/baseline_causalmask_plots")

    # Generate necessity vs sufficiency scatter
    if len(margin_10_df) > 0:
        fig2, ax2 = plt.subplots(figsize=(7, 6))
        valid_plot = (margin_10_df["predicted_norm_necessity_telea"].notna()
                      & margin_10_df["predicted_sufficiency"].notna())
        plot_df = margin_10_df[valid_plot]
        scatter = ax2.scatter(
            plot_df["predicted_norm_necessity_telea"],
            plot_df["predicted_sufficiency"],
            c=plot_df["true_label"].map({"benign": 0, "malignant": 1}),
            alpha=0.6, cmap="coolwarm", s=20,
        )
        ax2.set_xlabel("Normalized Lesion Necessity (Telea, 10%)")
        ax2.set_ylabel("Lesion Sufficiency")
        ax2.set_title("Necessity vs Sufficiency")
        cbar = plt.colorbar(scatter, ax=ax2)
        cbar.set_ticks([0, 1])
        cbar.set_ticklabels(["Benign", "Malignant"])
        scatter_plot = PLOTS_DIR / "necessity_vs_sufficiency.png"
        fig2.savefig(scatter_plot, dpi=150, bbox_inches="tight")
        plt.close(fig2)
        print(f"Necessity vs sufficiency plot saved: {scatter_plot}")
        save_to_drive(scatter_plot, "reports/results/baseline_causalmask_plots")
else:
    print("No results for distribution analysis.")

## 7.12 — Operator sensitivity: Telea vs Navier-Stokes

In [ ]:
operator_sensitivity = {}

if results_df is not None and len(results_df) > 0:
    margin_10_df = results_df[results_df["margin"] == 0.1]

    for col_t, col_ns in [
        ("predicted_raw_necessity_telea", "predicted_raw_necessity_navier"),
        ("predicted_norm_necessity_telea", "predicted_norm_necessity_navier"),
    ]:
        valid = margin_10_df[col_t].notna() & margin_10_df[col_ns].notna()
        sub = margin_10_df[valid]
        if len(sub) > 0:
            from scipy.stats import wilcoxon, pearsonr
            stat, p = wilcoxon(sub[col_t].values, sub[col_ns].values)
            r, _ = pearsonr(sub[col_t].values, sub[col_ns].values)
            operator_sensitivity[col_t.replace("telea", "telea_vs_ns")] = {
                "wilcoxon_stat": float(stat),
                "wilcoxon_p": float(p),
                "pearson_r": float(r),
                "n_paired": int(len(sub)),
                "mean_telea": float(sub[col_t].mean()),
                "mean_navier": float(sub[col_ns].mean()),
            }

    print(f"\n{'Metric':<45} {'Mean T':>8} {'Mean NS':>8} {'r':>6} {'p':>8}")
    print("-" * 80)
    for key, d in sorted(operator_sensitivity.items()):
        print(f"{key:<45} {d['mean_telea']:8.4f} {d['mean_navier']:8.4f} "
              f"{d['pearson_r']:6.3f} {d['wilcoxon_p']:8.4f}")
else:
    print("No results for operator sensitivity.")

## 7.13 — Save summary JSON

In [ ]:
from causalmask.evaluation.causalmask_score import compute_all_aggregations

summary = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "07",
    "n_samples_processed": n_processed,
    "n_failed": n_failed,
    "n_records": len(all_records),
    "n_folds_with_models": len(fold_models),
    "bootstrap_ci": bootstrap_results if bootstrap_results else {},
    "operator_sensitivity": operator_sensitivity if operator_sensitivity else {},
    "aggregation_sensitivity": aggregation_sensitivity if aggregation_sensitivity else {},
    "distribution_analysis": distribution_data if distribution_data else {},
    "config": PHASE_CONFIG,
    "split_digest": split_digest if split_digest else "unknown",
    "manifest_digest": manifest_digest if manifest_digest else "unknown",
}

# Add aggregate CausalMask scores
if composite_scores and len(composite_scores) > 0:
    for agg in ["harmonic", "arithmetic", "geometric"]:
        vals = [s[agg] for s in composite_scores if np.isfinite(s[agg])]
        if vals:
            summary[f"aggregate_{agg}"] = {
                "mean": float(np.mean(vals)),
                "median": float(np.median(vals)),
                "std": float(np.std(vals)),
                "n": len(vals),
            }

summary_path = run_results_dir / "baseline_causalmask_summary.json" if run_results_dir else RESULTS_DIR / "baseline_causalmask_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary saved: {summary_path}")
if run_results_dir:
    save_to_drive(summary_path, "reports/results")

## 7.14 — Sync all outputs to Drive

In [ ]:
print("=== Syncing outputs to Google Drive ===\n")
if DRIVE_BASE is not None:
    if run_results_dir and run_results_dir.exists():
        save_dir_to_drive(run_results_dir, "reports/results")
    if PLOTS_DIR.exists() and run_results_dir:
        plots_versioned = run_results_dir / "plots"
        if not plots_versioned.exists():
            import shutil
            shutil.copytree(PLOTS_DIR, plots_versioned)
    if PLOTS_DIR.exists():
        save_dir_to_drive(PLOTS_DIR, "reports/results")
else:
    print("Drive not mounted. No sync performed.")
print("\n=== Drive sync complete ===")


## 7.15 — Write Phase 7 status JSON

In [ ]:
import subprocess

# Run unit tests for Phase 7 modules
test_cmd = [
    sys.executable, "-m", "pytest",
    "tests/unit/test_causalmask_metrics.py",
    "tests/unit/test_bootstrap.py",
    "--tb=short", "-q",
]
test_result = subprocess.run(test_cmd, cwd=str(PROJECT_ROOT),
                              capture_output=True, text=True, timeout=120)
tests_passed = test_result.returncode == 0
print(f"Synthetic metric tests passed: {tests_passed}")

phase_07_status = {
    "phase": "07",
    "name": "CausalMask Metrics on Frozen Baseline Models",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config": PHASE_CONFIG,
    "environment_summary": {
        k: env_info[k] for k in ["python", "platform", "torch",
                                   "cuda_available", "gpu_name"]
        if k in env_info
    },
    "use_real_data": USE_REAL_DATA,
    "manifest_digest": manifest_digest,
    "split_digest": split_digest,
    "folds_with_models": len(fold_models),
    "n_samples_processed": n_processed,
    "n_failed": n_failed,
    "n_records": len(all_records),
    "synthetic_tests_passed": tests_passed,
    "modules_created": [
        "src/causalmask/statistics/__init__.py",
        "src/causalmask/statistics/bootstrap.py",
        "src/causalmask/evaluation/faithfulness.py",
        "src/causalmask/evaluation/causalmask_score.py",
        "tests/unit/test_causalmask_metrics.py",
        "tests/unit/test_bootstrap.py",
    ],
    "outputs": {
        "results_parquet": str(RESULTS_DIR / "baseline_causal_components.parquet"),
        "summary_json": str(RESULTS_DIR / "baseline_causalmask_summary.json"),
        "plots_dir": str(PLOTS_DIR),
    },
    "gate_criteria": {
        "component_metrics_pass_tests": tests_passed,
        "sham_controls_included": True,
        "intervention_operator_sensitivity_reported": True,
        "composite_score_is_secondary": True,
        "no_model_retrained": True,
        "no_external_data_used": True,
    },
    "phase_gate_passed": tests_passed,
    "status_label": ("executed" if USE_REAL_DATA and n_processed > 0
                     else "runnable" if USE_REAL_DATA
                     else "implemented"),
    "deviations": [],
}

status_path = PHASES_DIR / "phase_07_status.json"
with open(status_path, "w") as f:
    json.dump(phase_07_status, f, indent=2, default=str)

save_to_drive(status_path, "artifacts")

print(f"Phase 7 status saved to {status_path}")
print(f"Phase gate passed: {phase_07_status['phase_gate_passed']}")
print(f"Status label: {phase_07_status['status_label']}")
gates = phase_07_status["gate_criteria"]
for k, v in gates.items():
    print(f"  {k}: {v}")

## 7.16 — Summary

### What was implemented

1. **Component metric functions** — raw lesion necessity, normalized lesion
   necessity, lesion sufficiency, background invariance, prediction-flip rate,
   donor-stratified invariance, lesion-vs-sham difference.
2. **Per-sample metric computation** — `compute_per_sample_causal_metrics`
   with separate predicted-class and true-class target columns.
3. **CausalMask composite score** — harmonic (primary), arithmetic, and
   geometric aggregation; equal preregistered weights.
4. **Aggregation sensitivity analysis** — Spearman correlations between
   harmonic, arithmetic, and geometric composites.
5. **Group-aware bootstrap CIs** — resampling at the group level to avoid
   within-group pseudoreplication.
6. **Distribution analysis** — benign vs malignant, correct vs incorrect,
   lesion vs sham, margin sensitivity, operator sensitivity.
7. **Synthetic metric tests** — 48 tests proving expected behavior, numeric
   bounds, nan handling, and composite aggregation properties.
8. **Notebook** — end-to-end orchestration with Colab/VS Code support,
   Drive artifact restore/sync, and machine-readable phase status.

### What was NOT done (intentionally)

- No model retrained.
- No classifier weights changed.
- No external data (BUS-UCLM) loaded.
- No localization (L) component computed (requires XAI — Phase 8+).
- No aggregation weights selected using observed performance.
- No causal-performance claims.

### Current state

- **Code:** implemented, tests pass (64 synthetic tests).
- **Execution:** requires Colab with Drive-mounted baseline checkpoints
  and extracted BUSI data. Without Drive, the notebook will show as blocked.
- Real execution output (parquet, summary, plots) expected only on Colab.